# A3 — Image description + subclassification
Qwen3.5-4B (vision) and Qwen2.5-1.5B-Instruct (classification). Run on a Colab T4. Models are loaded **sequentially** to reduce VRAM pressure.

In [ ]:
%pip install -q -U "transformers>=4.57" accelerate safetensors pillow pandas pytest timm

If dependencies changed, restart the Colab runtime before continuing. Do not install `torchao` for this workflow.

In [ ]:
from pathlib import Path
import sys
GITHUB_REPO = "SunJ2432/ai-image-compliance"
repo_dir = Path("/content") / GITHUB_REPO.split("/")[-1]
if not repo_dir.exists():
    !git clone https://github.com/{GITHUB_REPO}.git {repo_dir}
else:
    !git -C {repo_dir} pull --ff-only
assert (repo_dir / "samples_tiny/manifest.csv").is_file(), "Dataset missing; check GitHub sync"
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))
print("Repo:", repo_dir)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Select T4 GPU under Runtime > Change runtime type"

## Stage 1 — Caption all images. The Qwen3.5-4B model downloads on first call. The identity-safety instruction is in `captioner/model.py`.

In [ ]:
import pandas as pd
from imgcompliance.captioner import Captioner
manifest = repo_dir / "samples_tiny/manifest.csv"
df = pd.read_csv(manifest)
captioner = Captioner()
caption_rows = []
for _, row in df.iterrows():
    path = manifest.parent / row["filename"]
    assert path.exists(), f"Missing image: {path}"
    caption = captioner.describe(path)
    caption_rows.append({"filename": row["filename"], "ground_truth": row["label"], "caption": caption})
    print(path.name, caption["scene_summary"][:120])

## Release the VLM before loading the classifier

In [ ]:
import gc
del captioner
gc.collect()
torch.cuda.empty_cache()
print("Captioning finished; GPU cache cleared")

## Stage 2 — Restore your A2 checkpoint and classify. If checkpoint is missing after a runtime reset, restore it or rerun A2 training first. The default context does **not** assert that any depicted entity has been verified.

In [ ]:
from imgcompliance.detector import Detector
from imgcompliance.classifier import Classifier
checkpoint = Path("/content/detector_head.pt")
assert checkpoint.exists(), "Restore your A2 checkpoint to /content/detector_head.pt"
detector = Detector(checkpoint=checkpoint)
classifier = Classifier()
results = []
for item in caption_rows:
    path = manifest.parent / item["filename"]
    detection = detector.predict(path)
    context = {"declared_purpose": None, "verified_real_entity": False}
    sub = classifier.classify(item["caption"], detection, context)
    results.append({
        "filename": item["filename"], "ground_truth": item["ground_truth"],
        "detected": detection["label"], "ai_score": round(detection["prob"], 4),
        "scene_summary": item["caption"]["scene_summary"],
        "depicts_people": item["caption"]["depicts_people"],
        "appears_photorealistic": item["caption"]["appears_photorealistic"],
        "possible_real_world_reference": item["caption"]["possible_real_world_reference"],
        "caption_format": item["caption"]["format"],
        "subclass": sub["subclass"], "rationale": sub["rationale"]
    })
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("/content/a3_results.csv", index=False)
print("Saved /content/a3_results.csv")

## Unit tests (no model downloads)

In [ ]:
!PYTHONPATH={repo_dir}/src python -m pytest -q {repo_dir}/tests/test_understand.py

**Interpretation:** This is a smoke test using the same six images as A2 training. It does not establish real-world accuracy. A deepfake subclass needs external evidence of a real-world entity; the landscape samples should generally be `general_synthetic` when the detector says `ai`.